In [ ]:
from jax import numpy as jnp, Array
from flax import nnx
from flax.nnx.module import first_from
from flax.typing import Shape

def softmax_cross_attention(q: jnp.ndarray, k: jnp.ndarray, v: jnp.ndarray, is_causal: bool):
    T_q, T_k = q.shape[-3], k.shape[-3]
    d_head = q.shape[-1]
    
    causal_mask = jnp.where(
        jnp.tri(T_q, T_k, T_k - T_q) if is_causal else jnp.ones((T_q, T_k)),
        0,
        -jnp.inf
    )

    attn_scores = jnp.einsum("...qhd,...khd->...hqk", q, k) / d_head ** 0.5
    attn_weights = nnx.softmax(attn_scores + causal_mask)

    return jnp.einsum("...hqk,...khd->...qhd", attn_weights, v)

class MultiHeadAttention(nnx.Module):
    def __init__(self, d_model: int, d_head: int, *, rngs: nnx.Rngs, decode = False):
        assert d_model % d_head == 0
        n_head = d_model // d_head

        self.decode = decode

        projection = lambda : nnx.Einsum(
            "...e,ehd->...hd",
            (d_model, n_head, d_head),
            (n_head, d_head),
            rngs=rngs
        )

        self.W_q = projection()
        self.W_k = projection()
        self.W_v = projection()

        self.W_o = nnx.Einsum(
            "...hd,hde->...e",
            (n_head, d_head, d_model),
            (d_model,),
            rngs=rngs
        )

        self.cached_key: None | nnx.Cache[Array] = nnx.data(None)
        self.cached_value: None | nnx.Cache[Array] = nnx.data(None)
        self.cache_index: None | nnx.Cache[Array] = nnx.data(None)

    def init_cache(self, cache_shape: Shape):
        self.cached_key = nnx.Cache(jnp.zeros(cache_shape))
        self.cached_query = nnx.Cache(jnp.zeros(cache_shape))
        self.cached_index = nnx.Cache(jnp.array(0, dtype=jnp.int32))

    def __call__(self, X: jnp.ndarray, memory: jnp.ndarray | None = None, is_causal = False):
        if memory is None:
            memory = X

        q = self.W_q(X)
        k = self.W_k(memory)
        v = self.W_v(memory)

        y = softmax_cross_attention(q, k, v, is_causal)

        return self.W_o(y)

In [ ]:
# Goal for tomorrow
# 1. Make some notes on the behaviour below
# 2. Finish cache implementation
# 3. Beyond that look at GPT-3


# 1. Updates the existing variable's array.
#    Cannot change its shape (although broadcasting may be allowed).
self.my_variable[...] = new_array

# 2. Replaces the underlying array, preserving variable identity.
#    Can change shape.
#    May trigger JIT recompilation if shape/dtype changes.
self.my_variable.set_value(new_array)

# 3. Replaces the Variable object itself.
#    Changes object identity and potentially graph structure/state.
#    May trigger JIT recompilation if the effective graph structure
#    or array shape/dtype changes.
self.my_variable = nnx.Variable(new_array)